### Prueba de script para descarga de datos de Global Fishing Watch  

In [1]:
#Bibliotecas a utilizar en este código
import pandas as pd # para la manipulación de tablas 
import geopandas as gpd #para la manipulación de objetos vectoriales
import matplotlib.pyplot as plt #para la creación de gráfica
#import gfwapiclient as gfw se explica más adelante porque es el objetivo del script


In [2]:
%pip install gfw-api-python-client

Note: you may need to restart the kernel to use updated packages.


In [3]:
#Biblioteca de GFW
import gfwapiclient as gfw

print("Cliente oficial de GFW instalado correctamente")

Cliente oficial de GFW instalado correctamente


In [4]:
#Importa la función "getpass" para la incorporación de información confidencial y que no se muestre al escribir
from getpass import getpass

In [5]:
GFW_TOKEN = getpass("Introduce tu API token de GFW: ")

Introduce tu API token de GFW:  ········


In [6]:
#Con esta función se crea un objeto que permite la comunicación con el servidor de Global Fishing Watch
gfw_client = gfw.Client(
    access_token=GFW_TOKEN
)

print("Conexión inicializada")

Conexión inicializada


In [7]:
# Para consultar las Zonas Económicas Exclusivas (ZEE en adelante) disponibles
ZEE = await gfw_client.references.get_eez_regions()

In [8]:
#Convertir los resultados de la búsqueda en una tabla o dataframe (df): 
ZEE = ZEE.df()
display(ZEE.head())

,id,label,iso3,dataset,isoSov1,isoSov2,isoSov3,territory1
0,8491,Jordanian Exclusive Economic Zone,JOR,public-eez-areas,JOR,None,None,Jordan
1,8467,Uruguayan Exclusive Economic Zone,URY,public-eez-areas,URY,None,None,Uruguay
2,5683,Latvian Exclusive Economic Zone,LVA,public-eez-areas,LVA,None,None,Latvia
3,48950,Overlapping claim Kuril Islands: Japan / Russia,None,public-eez-areas,JPN,RUS,None,Kuril Islands
4,8327,South Korean Exclusive Economic Zone,KOR,public-eez-areas,KOR,None,None,South Korea


In [9]:
#Filtrado de la tabla con la etiqueta del país: 
ZEE_JPN = ZEE[
    ZEE["label"].str.contains("Japan", case=False, na=False)
]

display(ZEE_JPN)

,id,label,iso3,dataset,isoSov1,isoSov2,isoSov3,territory1
3,48950,Overlapping claim Kuril Islands: Japan / Russia,None,public-eez-areas,JPN,RUS,None,Kuril Islands
120,48955,Overlapping claim Liancourt Rocks: Japan / Sou...,None,public-eez-areas,JPN,KOR,None,Liancourt Rocks
217,48954,Overlapping claim Senkaku Islands: Taiwan / Ja...,None,public-eez-areas,TWN,JPN,CHN,Senkaku Islands
231,8487,Japanese Exclusive Economic Zone,JPN,public-eez-areas,JPN,None,None,Japan
270,21796,Joint regime area: South Korea / Japan,None,public-eez-areas,KOR,JPN,None,South Korea


In [10]:
#Ahora se acota la búsqueda a la región de interés: el atolón coralino de Okinotorishima
filtro_okinotorishima = ZEE.astype(str).apply(
    lambda columna: columna.str.contains(
        "Okinotorishima",
        case=False,
        na=False
    )
).any(axis=1)

OKINOTORISHIMA = ZEE[filtro_okinotorishima]

display(OKINOTORISHIMA)

,id,label,iso3,dataset,isoSov1,isoSov2,isoSov3,territory1


In [11]:
#Como en la tabla no se muestran resultados, es posible que no exista bajo ese nombre o que debido a su reducido tamaño no esté considerado. 
# Para asegurarse de que no es una falla y que el resultado esté vacío 

print("Número de registros:", len(OKINOTORISHIMA))
print("Dimensiones:", OKINOTORISHIMA.shape)

Número de registros: 0
Dimensiones: (0, 8)


In [14]:
#El resultado confirma la ausencia de información para esa zona en las ZEE. De modo que se procede a dirigir la búsqueda a áreas marinas protegidas (AMP), auqnue es un territorio en disputa.
AMP_general= await gfw_client.references.get_mpa_regions()

In [15]:
#Convertir resultado a data frame

AMP = AMP_general.df()

display(AMP.head())

,id,label,name,dataset
0,1,Diamond Reef - Marine Reserve,None,public-mpa-all
1,2,Palaster Reef - Marine Reserve,None,public-mpa-all
2,27,Folkstone - Marine Reserve,None,public-mpa-all
3,46,Reserva Biológica Do Atol Das Rocas - Reserva ...,None,public-mpa-all
4,57,Parque Nacional Do Cabo Orange - Parque,None,public-mpa-all


In [16]:
#Comprobación de la información: 
print("Número de registros:", len(AMP))
print("Columnas:", AMP.columns.tolist())

Número de registros: 17514
Columnas: ['id', 'label', 'name', 'dataset']


In [17]:
#Aplicación de filtrado para buscar Okinotorishima en las AMP oficiales

filtro_okinotorishima = AMP.astype(str).apply(
    lambda columna: columna.str.contains(
        "Okinotorishima",
        case=False,
        na=False
    )
).any(axis=1)

AMP_OKINOTORISHIMA = AMP[filtro_okinotorishima]

display(AMP_OKINOTORISHIMA)

,id,label,name,dataset


In [19]:
#Conclusión: no existe un registro oficial del atolón como área marina protegida, se propone redireccionar la búsqueda a AMPs de Japón. 

In [20]:
# Filtrar áreas protegidas que mencionen Japón

AMP_JAPON = AMP[
    AMP["label"].str.contains(
        "Japan",
        case=False,
        na=False
    )
].copy()

display(AMP_JAPON)

print("Total de AMP encontradas:", len(AMP_JAPON))

,id,label,name,dataset


Total de AMP encontradas: 0


In [22]:
#Como no se encontró ninguna AMP asociada con el nombre del país, se procede a hacer una búsqueda rápida en internet y utilizar el nombre de un AMP 
# relacionada con actividad pesquera
# Identificar el Parque Nacional Shiretoko en el catálogo GFW

AMP_SHIRETOKO = AMP[
    AMP.astype(str).apply(
        lambda columna: columna.str.contains(
            "Shiretoko",
            case=False,
            na=False
        )
    ).any(axis=1)
].copy()

display(AMP_SHIRETOKO)

,id,label,name,dataset


In [ ]:
#Una vez más, no arroja nada la búsqueda. Nueva propuesta: cambiar la región de interés a una de Latinoamérica o de un sitio del cual tengamos un
# mejor manejo del idioma.

In [24]:
#Prueba con México

filtro_MEX = AMP.astype(str).apply(
    lambda columna: columna.str.contains(
        "Mexico",
        case=False,
        na=False
    )
).any(axis=1)

AMP_MEX = AMP[filtro_MEX]

display(AMP_MEX)

,id,label,name,dataset
2413,140319,Gwithian to Mexico Towans - Site of Special Sc...,None,public-mpa-all


In [25]:
# Los resultados muestran la falla en el codigo de búsqueda porque no muestra AMPs de México. Se procede a buscar directamente con los nombres de AMPs
# que además son Reservas de la Biosfera, asumiendo que por declaracion internacional mejore la búsqueda
# Buscar Banco Chinchorro en el catálogo de AMP

filtro_chinchorro = AMP.astype(str).apply(
    lambda columna: columna.str.contains(
        "Banco Chinchorro",
        case=False,
        na=False
    )
).any(axis=1)

AMP_CHINCHORRO = AMP[filtro_chinchorro].copy()

display(AMP_CHINCHORRO)

,id,label,name,dataset
1840,103171,Banco Chinchorro - Reserva de la Biósfera,None,public-mpa-all
6376,902304,Reserva de la Biosfera Banco Chinchorro - Siti...,None,public-mpa-all


In [26]:
#Resultados: Ya se muestra la información para el área indicada. Conclusión: seleccionesmos un AMP de la cual conozcamos el nombre :)